# Search Ergonomics (FTS + Vector)

Release-focused examples for preferred query APIs:
- Full-text: `search`, `search_and`, `search_or`, score/highlight projections
- Vector: `semantic_search` and legacy-compatible `__knn`
- Hybrid composition via two-phase intersection (portable across runtimes)

## What you should see
- FTS queries compile and execute (no missing analyzer errors).
- Search query returns rows from seeded demo data.
- Final cleanup disconnects the DB session.


In [ ]:
from surrealengine import Document, StringField, VectorField, create_connection

db = create_connection(url="memory://", namespace="demo", database="search_ergonomics", make_default=True)
await db.connect()

# Ensure a portable default analyzer exists for FULLTEXT demos.
for stmt in [
    "DEFINE ANALYZER ascii TOKENIZERS blank,punct FILTERS lowercase",
    "DEFINE ANALYZER ascii TOKENIZERS blank FILTERS lowercase",
    "DEFINE ANALYZER ascii TOKENIZERS blank",
    "DEFINE ANALYZER ascii",
]:
    try:
        await db.client.query(stmt)
        break
    except Exception:
        pass


In [ ]:
class Article(Document):
    title = StringField()
    content = StringField()
    embedding = VectorField(dimension=4)

    class Meta:
        collection = "articles_search_erg"
        indexes = [
            {
                "name": "idx_title",
                "fields": ["title"],
                "search": True,
                "bm25": True,
                "highlights": True,
            },
            {
                "name": "idx_content",
                "fields": ["content"],
                "search": True,
                "bm25": True,
                "highlights": True,
            },
            {
                "name": "idx_embedding",
                "fields": ["embedding"],
                "dimension": 4,
                "dist": "cosine",
                "m": 16,
                "efc": 100,
            },
        ]

await Article.create_table()


In [ ]:
# Seed data so query execution has visible results
try:
    await db.client.query("REMOVE TABLE articles_search_erg")
except Exception:
    pass

await Article.create_table()

await db.client.query("DELETE articles_search_erg")

await Article(title="Hybrid Retrieval", content="hybrid retrieval pipelines", embedding=[0.91, 0.07, 0.01, 0.01]).save()
await Article(title="Vector Search", content="dense vectors and semantic ranking", embedding=[0.9, 0.05, 0.03, 0.02]).save()
await Article(title="Keyword Search", content="bm25 full text ranking", embedding=[0.1, 0.7, 0.1, 0.1]).save()

seeded = await Article.objects.all()
print("seeded rows:", len(seeded))


In [ ]:
# Preferred vector API
q_vec = (
    Article.objects
    .semantic_search(field="embedding", vector=[0.91, 0.07, 0.01, 0.01], k=10, metric="COSINE")
    .limit(10)
)
print(q_vec.get_raw_query())

# Legacy-compatible vector operator
q_knn = Article.objects.filter(embedding__knn=([0.91, 0.07, 0.01, 0.01], 10, "COSINE"))
print(q_knn.get_raw_query())


In [ ]:
# FTS helpers
q_fts = (
    Article.objects
    .search_and("agent memory", "content")
    .with_search_score(reference=1, alias="score")
    .with_search_highlight("<b>", "</b>", reference=1, alias="snippet")
    .order_by("score", "DESC")
    .limit(5)
)
print(q_fts.get_raw_query())

q_or = Article.objects.search_or("hybrid retrieval", "title", "content")
print(q_or.get_raw_query())

d = await q_or.all()
print("q_or rows:", len(d))
for item in d:
    print(item.id)


In [ ]:
# Hybrid composition example (portable two-phase approach)
# Note: some runtimes do not reliably support KNN + FTS in one WHERE clause.
# We intersect IDs from each modality to keep behavior deterministic.

fts_docs = await Article.objects.search("retrieval", "content").all()
fts_ids = {str(doc.id) for doc in fts_docs}
print("fts ids:", fts_ids)

vec_docs = await Article.objects.semantic_search(
    field="embedding",
    vector=[0.9, 0.05, 0.03, 0.02],
    k=20,
    metric="COSINE",
).all()

hybrid_docs = [doc for doc in vec_docs if str(doc.id) in fts_ids][:10]
print("hybrid rows:", len(hybrid_docs), [doc.title for doc in hybrid_docs])


In [ ]:
await db.disconnect()
